## Libs

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
import numpy as np
from sklearn import metrics
from sklearn.datasets import make_moons


from tqdm import tqdm

import optuna
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

d:\programming\26.1\IAP\MLP_implementation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando dispositivo: cuda


## 1. Classe do MLP

In [ ]:
class MLP(nn.Module):
    def __init__(
        self,
        input_size,
        num_classes,
        hidden_sizes,
        dropout_rate,
        learning_rate,
        momentum,
        activation_func,
        optimizer_class,
    ):
        super(MLP, self).__init__()
        layers = []
        in_layer = input_size
        self.learning_rate = learning_rate
        self.momentum = momentum
        
        # Construir camadas da rede
        for h_size in hidden_sizes:
            layers.append(nn.Linear(in_layer, h_size))
            layers.append(activation_func)
            layers.append(nn.Dropout(dropout_rate))
            in_layer = h_size

        # Camada de saída
        layers.append(nn.Linear(in_layer, num_classes))
        self.model = nn.Sequential(*layers)

        if optimizer_class == torch.optim.SGD:
            self.optimizer = optimizer_class(
                self.parameters(), lr=learning_rate, momentum=momentum
            )
        else:
            self.optimizer = optimizer_class(self.parameters(), lr=learning_rate)

    def forward(self, x):
        return self.model(x)

    def get_optimizer(self):
        return self.optimizer

## Funções para Treinamento e Tunning

### Treino por época

In [5]:
def train_epoch(model, dataloader, loss_func, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        model.optimizer.zero_grad()
        outputs = model(X_batch)
        loss = loss_func(outputs, y_batch)
        loss.backward()
        model.optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)


### Validação

In [4]:
def validate(model, dataloader, loss_func, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = loss_func(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            all_preds.append(outputs.cpu())
            all_labels.append(y_batch.cpu())
    avg_loss = total_loss / len(dataloader.dataset)
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    return avg_loss, all_preds, all_labels

### Tunning

In [ ]:


def create_objective(X: torch.Tensor, y: torch.Tensor, num_epochs: int, device: torch.device):
    hyperparam_dict = {
        "hidden_sizes": [[64], [128], [64, 32], [128, 64]],
        "dropout_rate": (0.0, 0.5),
        "learning_rate": (1e-4, 1e-1),
        "momentum": (0.0, 0.9),
        "activation_func": [nn.ReLU(), nn.Tanh(), nn.Sigmoid()],
        "optimizer_class": [torch.optim.SGD, torch.optim.Adam],
    }
    # converter para array numpy para usar com KFold


    def objective(trial):
        # Hiperparâmetros a serem otimizados
        hidden_sizes = trial.suggest_categorical("hidden_sizes", hyperparam_dict["hidden_sizes"])
        learning_rate = trial.suggest_loguniform("learning_rate", *hyperparam_dict["learning_rate"])
        momentum = trial.suggest_float("momentum", *hyperparam_dict["momentum"])
        dropout_rate = trial.suggest_float("dropout_rate", *hyperparam_dict["dropout_rate"])
        activation_func = trial.suggest_categorical("activation_func", hyperparam_dict["activation_func"])
        optimizer_class = trial.suggest_categorical("optimizer_class", hyperparam_dict["optimizer_class"])

        # Validação cruzada
        k = 5
        folds = KFold(n_splits=k, shuffle=True, random_state=42)
        val_losses = []

        for fold, (train_idx, val_idx) in enumerate(folds.split(X)):
            
            # modelo e função de custo
            model = MLP(
                input_size=X.shape[1],
                num_classes=len(torch.unique(y)),
                hidden_sizes=hidden_sizes,
                dropout_rate=dropout_rate,
                learning_rate=learning_rate,
                momentum=momentum,
                activation_func=activation_func,
                optimizer_class=optimizer_class,
            ).to(device)

            loss_func = nn.CrossEntropyLoss()

            # dados
            train_subset = Subset(TensorDataset(
                    X_tensor[train_idx], 
                    y_tensor[train_idx]), 
                    range(len(train_idx))
                    )
            val_subset = Subset(TensorDataset(
                X_tensor[val_idx], 
                y_tensor[val_idx]), 
                range(len(val_idx))
                )
            train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_subset, batch_size=32)


            for epoch in range(num_epochs):
                train_epoch(model, train_loader, loss_func, device)

            # Validar no fold atual
            val_loss_fold, _, _ = validate(model, val_loader, loss_func, device)
            val_losses.append(val_loss_fold)
        
        # Retornar a média das perdas de validação
        return np.mean(val_losses)  
         
    return objective

                

## Otimização

In [ ]:

# test with make mooons first
X_train, y_train = make_moons(n_samples=100, noise=0.1, random_state=42)
X_test, y_test = make_moons(n_samples=20, noise=0.2, random_state=43)
# Converter dados para tensores
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

study = optuna.create_study(direction="minimize")
objective_fun = create_objective(X_tensor, y_tensor, num_epochs=3, device=device)
study.optimize(objective_fun, n_trials=2)

[I 2026-04-05 23:58:14,009] A new study created in memory with name: no-name-92487963-f0d1-4e2f-9b04-097f837374b2
d:\programming\26.1\IAP\MLP_implementation\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [64] which is of type list.
  optuna_warn(message)
d:\programming\26.1\IAP\MLP_implementation\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [128] which is of type list.
  optuna_warn(message)
d:\programming\26.1\IAP\MLP_implementation\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [64, 32] which is of type list.
  optuna_warn(message)
d:\programming\26.1\

TypeError: _unique2(): argument 'input' (position 1) must be Tensor, not numpy.ndarray